In [14]:
import os
from dotenv import load_dotenv
from langchain.vectorstores import Pinecone as LangChainPinecone
from langchain.embeddings import HuggingFaceEmbeddings
from pinecone import Pinecone
from langchain_google_genai import ChatGoogleGenerativeAI
import google.generativeai as genai
import json
# from recipe_filter import filter_allergens_in_variants, filter_and_sort_recipes
import ast

# Load environment variables
load_dotenv(override=True)

gemini_api_key = os.getenv('GOOGLE_API_KEY')

# Ensure your Google API key is set
genai.configure(api_key=gemini_api_key)

# Initialize Pinecone
pc = Pinecone(api_key=os.getenv('PINECONE_API_KEY'))
index_name = "recipes-index"

# Load the embedding model (same as used for storing data)
embed_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# Connect Pinecone to LangChain
vectorstore = LangChainPinecone(pc.Index(index_name), embed_model, text_key="text")

# Initialize ChatGoogleGenerativeAI for gemini-1.5-flash
llm = ChatGoogleGenerativeAI(
    model="gemini-1.5-flash",
    temperature=0,
    max_tokens=None,
    timeout=None,
    max_retries=2,
)

# Function to generate responses
def generate_response(prompt):
    model = genai.GenerativeModel("gemini-1.5-flash")
    # Generate content based on the prompt
    response = model.generate_content(prompt)
    return response.text

# Function to filter and sort recipes
def filter_recipes(vectorstore, user_allergens, user_dislikes, query, meal_category, top_k):
    pinecone_filter = {
        "meal_category": {"$eq": meal_category},  # Filter for specific meal type
        "allergens": {"$nin": list(user_allergens)},  # Exclude recipes containing allergens
        "ingredients": {"$nin": list(user_allergens)}  # Exclude recipes containing allergens
    }

    # Retrieve documents from Pinecone with filtering for dislikes and allergens in ingredients
    docs = vectorstore.similarity_search(
        query=query,
        k=top_k,  # Fetch only the required number of results
        filter=pinecone_filter  # Apply the filter for dislikes and allergens in ingredients
    )


    return docs


# Function to format the filtered recipes into a structured meal plan prompt
def format_meal_plan_prompt(filtered_docs, query):
    prompt = f"""Generate a weekly meal plan based on the following user persona and recipe data:
    Only use the recipes exactly as they are presented below. Do not modify the recipes, add any extra information, or create new recipes. 
    Simply output the meal name for each day, using only the recipes provided. Do not alter or adapt the meals.
    May include those recipes that have disliked ingredients only if the recipes are not sufficient.
    If the available recipes are insufficient, you may include recipes that contain disliked ingredients. However, you must strictly follow these rules when checking for disliked ingredients:

    1. **ONLY check for disliked ingredients listed in the user's dislikes below. Do NOT flag any ingredient that is NOT in the dislikes list.**
    2. **You must NOT infer, assume, or guess the presence of any ingredient. Only check the explicit list of ingredients provided in the recipe.**
    3. **If a disliked ingredient (from the list below) is found in a meal, append the following notation to the meal: ' * (contains <disliked ingredient>)'.**
    4. **If a meal does not contain any disliked ingredients from the list below, do NOT append anything.**
    5. **Do NOT flag Butter, Cheddar Cheese, Yoghurt, Mascarpone Cheese, Mayonnaise, or any other ingredient unless they are explicitly listed in the dislikes section below and the ingredients list of the recipe.**
    6. **Ensure that the output is in JSON format only.**

    Give response in json format only.
        User Persona:
         Dietary Restrictions: {user_allergens}
         Dislikes: {user_dislikes}
         Likes: Mediterranean,European cuisine,Comfort food 
         Spice Level: Medium
         Age: 29
         Gender: Male
         Height: 180 cm
         Weight: 75 kg 
         Popular Dishes: Classic Chicken Salad, Crudites & Sour Cream Dip, Mini Quiches, Omega Egg Protein Pot
         Meal Frequency: 4 meals per day
         Meal Timing: Breakfast, Lunch, Dinner,and 2 snacks
        """
    for i, doc in enumerate(filtered_docs, 1):
        metadata = doc.metadata

        prompt += f"Meal {i}:\n"
        prompt += f" Dish Name: {metadata.get('dish_name', 'Unknown')}\n"
        prompt += f" Description: {metadata.get('description', 'No description')}\n"
        prompt += f" Ingredients: {', '.join(metadata.get('ingredients', []))}\n"
        prompt += f" Spice Level: {metadata.get('spice_level', 'Not specified')}\n"
        prompt += f" Cuisine: {metadata.get('cuisine', 'Unknown')}\n"
        prompt += f" Meal Category: {metadata.get('meal_category', 'Unknown')}\n"
        # prompt += f" Variants: {metadata.get('variants', 'Unknown')}\n"
    
    return prompt

# Main function to generate the meal plan
def generate_meal_plan(vectorstore, user_allergens, user_dislikes, query):
   
    # Step 1: Fetch exact numbers of recipes for each category directly from Pinecone
    breakfast_docs = filter_recipes(vectorstore, user_allergens, user_dislikes, query, "breakfast", 8)
    meal_docs = filter_recipes(vectorstore, user_allergens, user_dislikes, query, "meal", 16)
    snack_docs = filter_recipes(vectorstore, user_allergens, user_dislikes, query, "snack", 16)

    # Step 2: Organize the filtered recipes by meal category
    # breakfast_docs, meal_docs, snack_docs = organize_recipes_by_category(docs_sorted)
      # Debugging: Print fetched recipe counts
    print(f"Breakfast recipes: {len(breakfast_docs)}")
    print(f"Meal recipes: {len(meal_docs)}")
    print(f"Snack recipes: {len(snack_docs)}")
    print("Fetching complete.")

    # Step 2: Combine all selected recipes into the final list
    final_docs = breakfast_docs + meal_docs + snack_docs

    # Step 3: Format the structured meal plan prompt
    final_prompt = format_meal_plan_prompt(final_docs, query)

    # Step 4: Generate the response using LLM
    meal_plan = generate_response(final_prompt)

    print(meal_plan)
    return meal_plan

# Example usage
if __name__ == "__main__":
    # User preferences (replace with dynamic input if needed)
    user_allergens = {}  # Example allergens
    user_dislikes = {
        "Cod (white fish)", "Cod Fish", "Cuttlefish", "Fish Sauce",
        "Gochujang Paste", "Gochujang Sauce", "Local Wild Fish",
        "Nile Perch", "Salmon", "Sea Bass", "Squid", "Tuna",
        "White Fish", "Worcestershire Sauce"
    }  # Example disliked ingredients
    
    query = "Spice Level: Medium, Cuisine: Mediterranean,European,Comfort Food. Popular Dishes: Classic Chicken Salad, Crudites & Sour Cream Dip, Mini Quiches, Omega Egg Protein Pot"

    # Generate the meal plan
    generate_meal_plan(vectorstore, user_allergens, user_dislikes, query)

Breakfast recipes: 8
Meal recipes: 16
Snack recipes: 14
Fetching complete.
```json
{
  "weeklyMealPlan": {
    "Monday": {
      "Breakfast": "Chicken Casserole",
      "Lunch": "Chicken Tenders",
      "Dinner": "English Breakfast",
      "Snack 1": "Yoghurt Granola * (contains Yoghurt)",
      "Snack 2": "Reuben Sandwich * (contains Worcestershire Sauce)"
    },
    "Tuesday": {
      "Breakfast": "Chocolate Crepes",
      "Lunch": "Savory Baked Beans & Tortilla",
      "Dinner": "Oatmeal Apple Pancake",
      "Snack 1": "Poke Bowl",
      "Snack 2": "Sumac Protein & Lemon Herb Couscous"
    },
    "Wednesday": {
      "Breakfast": "Ginger-Sesame Protein with Quinoa",
      "Lunch": "Kale Salad with Maple Tahina Dressing",
      "Dinner": "Jamaican Chili and Onion Bread Roll",
      "Snack 1": "Lime Chili Quesadillas",
      "Snack 2": "Mac & Cheese with Cauliflower"
    },
    "Thursday": {
      "Breakfast": "Eggplant Lasagne",
      "Lunch": "Chimichurri Steak",
      "Dinner": "C

In [7]:
# Step 1: Fetch exact numbers of recipes for each category directly from Pinecone
breakfast_docs = filter_recipes(vectorstore, user_allergens, user_dislikes, query, "breakfast", 8)
meal_docs = filter_recipes(vectorstore, user_allergens, user_dislikes, query, "meal", 16)
snack_docs = filter_recipes(vectorstore, user_allergens, user_dislikes, query, "snack", 16)

# Step 2: Organize the filtered recipes by meal category
# breakfast_docs, meal_docs, snack_docs = organize_recipes_by_category(docs_sorted)
    # Debugging: Print fetched recipe counts
print(f"Breakfast recipes: {len(breakfast_docs)}")
print(f"Meal recipes: {len(meal_docs)}")
print(f"Snack recipes: {len(snack_docs)}")
print("Fetching complete.")

# Step 2: Combine all selected recipes into the final list
final_docs = breakfast_docs + meal_docs + snack_docs

Breakfast recipes: 8
Meal recipes: 16
Snack recipes: 14
Fetching complete.


In [9]:
len(final_docs)

38

In [10]:
import pandas as pd


In [15]:
# Weekly meal plan provided
meal_plan = {

    "Monday": {
      "Breakfast": "Chicken Casserole",
      "Lunch": "Chicken Tenders",
      "Dinner": "English Breakfast",
      "Snack 1": "Yoghurt Granola * (contains Yoghurt)",
      "Snack 2": "Reuben Sandwich * (contains Worcestershire Sauce)"
    },
    "Tuesday": {
      "Breakfast": "Chocolate Crepes",
      "Lunch": "Savory Baked Beans & Tortilla",
      "Dinner": "Oatmeal Apple Pancake",
      "Snack 1": "Poke Bowl",
      "Snack 2": "Sumac Protein & Lemon Herb Couscous"
    },
    "Wednesday": {
      "Breakfast": "Ginger-Sesame Protein with Quinoa",
      "Lunch": "Kale Salad with Maple Tahina Dressing",
      "Dinner": "Jamaican Chili and Onion Bread Roll",
      "Snack 1": "Lime Chili Quesadillas",
      "Snack 2": "Mac & Cheese with Cauliflower"
    },
    "Thursday": {
      "Breakfast": "Eggplant Lasagne",
      "Lunch": "Chimichurri Steak",
      "Dinner": "Creamy Protein & Mash Potatoes",
      "Snack 1": "Maqluba",
      "Snack 2": "Beef Bourguignon"
    },
    "Friday": {
      "Breakfast": "Lentil Curry & Almond Black Rice",
      "Lunch": "Peri Peri Wrap",
      "Dinner": "Fusilli Alfredo",
      "Snack 1": "Mansaf * (contains Yoghurt)",
      "Snack 2": "Classic Chicken Salad"
    },
    "Saturday": {
      "Breakfast": "Crudites and Smoked Paprika Dip",
      "Lunch": "Tuna Salad",
      "Dinner": "Chicken Poppers * (contains Yoghurt)",
      "Snack 1": "Truffle & Chive Scones",
      "Snack 2": "Mushroom Soup"
    },
    "Sunday": {
      "Breakfast": "Cheese and Nuts Pot",
      "Lunch": "Jallab",
      "Dinner": "Kunafa Cup",
      "Snack 1": "Grapes Pot",
      "Snack 2": "Dates & Almond Smoothie"
    }

}

# Day mapping
day_mapping = {
    "Monday": 1,
    "Tuesday": 2,
    "Wednesday": 3,
    "Thursday": 4,
    "Friday": 5,
    "Saturday": 6,
    "Sunday": 7
}

# Function to get details from final_docs
def get_meal_details(dish_name):
    for doc in final_docs:
        if doc.metadata['dish_name'] == dish_name:
            return doc.metadata
    return {}

# Extract the meals and their details
meal_data = []

# Process each day's meal plan
for day, meals in meal_plan.items():
    day_number = day_mapping[day]
    for meal_type, meal_name in meals.items():
        meal_details = get_meal_details(meal_name)
        if meal_details:
            # Add meal type, day, and day number
            meal_details['meal_type'] = meal_type
            meal_details['day'] = day
            meal_details['day_number'] = day_number
            meal_data.append(meal_details)

# Convert the data into a DataFrame
df = pd.DataFrame(meal_data)

# Show the resulting DataFrame
df.head()

,allergens,cuisine,description,dish_name,dish_type,ingredients,meal_category,recipe_id,spice_level,variants,meal_type,day,day_number
0,"[Dairy, Eggs]",European,Classic Spanish frittata with chicken. Contain...,Chicken Casserole,[Chef's Choice],"[Chicken, Olive Oil, Salt, Pickled Gherkins, W...",breakfast,67972a3ffead7e2e6ad9e828,Low,"kcal: 310, carb: 17, fat: 15, protein: 25",Breakfast,Monday,1
1,"[Dairy, Eggs, Gluten, Mustard]",European,-,Chicken Tenders,[Chef's Choice],"[Salt, Sunflower Oil, Mayonnaise, Bell Pepper,...",breakfast,67a06b25c2be0ed8ea625dae,Medium,"kcal: 436, carb: 20, fat: 18, protein: 47",Lunch,Monday,1
2,"[Dairy, Eggs, Gluten, Soy]",European,"With fried egg, baked beans, mushrooms & sausa...",English Breakfast,[Chef's Choice],"[Olive Oil, Salt, Parsley, Sunflower Oil, Wate...",breakfast,67a1c699c2715c683a3cda0a,Medium,"kcal: 371, carb: 13, fat: 22, protein: 30",Dinner,Monday,1
3,"[Dairy, Eggs, Gluten, Nuts]",Mediterranean,"With homemade chocolate, mascarpone cream & st...",Chocolate Crepes,[Comfort Food],"[Dark Chocolate, Strawberries, Vanilla, Egg, S...",breakfast,66ba2a2a21ed723107b1d8df,Medium,"kcal: 331, carb: 37, fat: 16, protein: 8",Breakfast,Tuesday,2
4,"[Gluten, Mustard]",American,"Cooked in tomatoes, mushrooms & pickled caulif...",Savory Baked Beans & Tortilla,[Comfort Food],"[Salt, Parsley, Sunflower Oil, White Vinegar, ...",breakfast,6690f7ac3d6d34934a271853,Low,"kcal: 360, carb: 55, fat: 9, protein: 12",Lunch,Tuesday,2


In [16]:
# Identify the top cuisine preference
top_cuisine = df['cuisine'].mode()[0]
print(f"Top Cuisine: {top_cuisine}")

Top Cuisine: Mediterranean


In [17]:

# Get the count of each cuisine selected by the user
cuisine_counts = df['cuisine'].value_counts().reset_index()

# Rename columns for clarity
cuisine_counts.columns = ["Cuisine", "Count"]

# Display as a formatted table (this works in Jupyter)
display(cuisine_counts)

,Cuisine,Count
0,Mediterranean,12
1,European,7
2,American,3
3,Arabic,3
4,Comfort Food,2
5,South American,1
6,Fusion,1


In [32]:
import pandas as pd

# Weekly meal plan provided
meal_plan = {
    "Monday": {
        "Breakfast": "Chicken Casserole",
        "Lunch": "Chicken Tenders",
        "Dinner": "English Breakfast",
        "Snack 1": "Yoghurt Granola ",
        "Snack 2": "Reuben Sandwich * (contains Worcestershire Sauce)"
    },
    "Tuesday": {
        "Breakfast": "Chocolate Crepes",
        "Lunch": "Savory Baked Beans & Tortilla",
        "Dinner": "Oatmeal Apple Pancake",
        "Snack 1": "Poke Bowl",
        "Snack 2": "Sumac Protein & Lemon Herb Couscous "
    },
    "Wednesday": {
        "Breakfast": "Ginger-Sesame Protein with Quinoa",
        "Lunch": "Kale Salad with Maple Tahina Dressing",
        "Dinner": "Jamaican Chili and Onion Bread Roll",
        "Snack 1": "Lime Chili Quesadillas",
        "Snack 2": "Mac & Cheese with Cauliflower"
    },
    "Thursday": {
        "Breakfast": "Eggplant Lasagne",
        "Lunch": "Chimichurri Steak",
        "Dinner": "Creamy Protein & Mash Potatoes",
        "Snack 1": "Maqluba",
        "Snack 2": "Beef Bourguignon"
    },
    "Friday": {
        "Breakfast": "Lentil Curry & Almond Black Rice",
        "Lunch": "Peri Peri Wrap",
        "Dinner": "Fusilli Alfredo",
        "Snack 1": "Mansaf ",
        "Snack 2": "Classic Chicken Salad"
    },
    "Saturday": {
        "Breakfast": "Crudites and Smoked Paprika Dip",
        "Lunch": "Tuna Salad",
        "Dinner": "Chicken Poppers ",
        "Snack 1": "Truffle & Chive Scones",
        "Snack 2": "Mushroom Soup"
    },
    "Sunday": {
        "Breakfast": "Cheese and Nuts Pot",
        "Lunch": "Jallab",
        "Dinner": "Kunafa Cup",
        "Snack 1": "Grapes Pot",
        "Snack 2": "Dates & Almond Smoothie"
    }
}

# Day mapping
day_mapping = {
    "Monday": 1,
    "Tuesday": 2,
    "Wednesday": 3,
    "Thursday": 4,
    "Friday": 5,
    "Saturday": 6,
    "Sunday": 7
}

# Function to get details from final_docs
def get_meal_details(dish_name):
    # Debugging: Print dish_name
    # print(f"Searching for: {dish_name}")
    for doc in final_docs:
        if doc.metadata['dish_name'].strip().lower() == dish_name.strip().lower():
            return doc.metadata
    return {}

# Extract the meals and their details
meal_data = []

# Process each day's meal plan
for day, meals in meal_plan.items():
    day_number = day_mapping[day]
    for meal_type, meal_name in meals.items():
        # Clean up meal name: Remove anything after '*' and strip extra spaces
        clean_meal_name = meal_name.split('*')[0].strip()

        # Debugging: Print cleaned meal name
        # print(f"Cleaned meal name: {clean_meal_name}")

        meal_details = get_meal_details(clean_meal_name)
        if meal_details:
            # Add meal type, day, and day number
            meal_details['meal_type'] = meal_type
            meal_details['day'] = day
            meal_details['day_number'] = day_number
            meal_data.append(meal_details)

# Convert the data into a DataFrame
df = pd.DataFrame(meal_data)

# Show the resulting DataFrame
df.head()


,allergens,cuisine,description,dish_name,dish_type,ingredients,meal_category,recipe_id,spice_level,variants,meal_type,day,day_number
0,"[Dairy, Eggs]",European,Classic Spanish frittata with chicken. Contain...,Chicken Casserole,[Chef's Choice],"[Chicken, Olive Oil, Salt, Pickled Gherkins, W...",breakfast,67972a3ffead7e2e6ad9e828,Low,"kcal: 310, carb: 17, fat: 15, protein: 25",Breakfast,Monday,1
1,"[Dairy, Eggs, Gluten, Mustard]",European,-,Chicken Tenders,[Chef's Choice],"[Salt, Sunflower Oil, Mayonnaise, Bell Pepper,...",breakfast,67a06b25c2be0ed8ea625dae,Medium,"kcal: 436, carb: 20, fat: 18, protein: 47",Lunch,Monday,1
2,"[Dairy, Eggs, Gluten, Soy]",European,"With fried egg, baked beans, mushrooms & sausa...",English Breakfast,[Chef's Choice],"[Olive Oil, Salt, Parsley, Sunflower Oil, Wate...",breakfast,67a1c699c2715c683a3cda0a,Medium,"kcal: 371, carb: 13, fat: 22, protein: 30",Dinner,Monday,1
3,"[Dairy, Nuts]",American,"With mixed berries, nuts & honey. Contains Dairy.",Yoghurt Granola,[Comfort Food],"[Mixed Berries, Raisins, Stevia, Water, Lemon ...",breakfast,6705438493a773db3a3dce78,Low,"kcal: 323, carb: 42, fat: 11, protein: 11",Snack 1,Monday,1
4,"[Dairy, Eggs, Fish, Gluten, Mustard, Soy]",American,"English muffins with turkey ham, eggs & chedda...",Reuben Sandwich,[Sandwich],"[Whole Spice, Salt, Sunflower Oil, Water, Vine...",breakfast,670555e10c299961d93ea65d,Low,"kcal: 480, carb: 35, fat: 22, protein: 33",Snack 2,Monday,1


In [33]:
# Identify the top cuisine preference
top_cuisine = df['cuisine'].mode()[0]
print(f"Top Cuisine: {top_cuisine}")

Top Cuisine: Mediterranean


In [34]:
# Get the count of each cuisine selected by the user
cuisine_counts = df['cuisine'].value_counts().reset_index()

# Rename columns for clarity
cuisine_counts.columns = ["Cuisine", "Count"]

# Display as a formatted table (this works in Jupyter)
display(cuisine_counts)

,Cuisine,Count
0,Mediterranean,13
1,European,9
2,American,5
3,Arabic,4
4,Comfort Food,2
5,South American,1
6,Fusion,1
